# 04 — Reproduce H→bb Baseline from Repository

Run the existing `stxs_functions.stxs_fit` as control/reference for H→bb-like two-bin setup.

In [ ]:
from pathlib import Path
import json, sys, os, math
import numpy as np
import pandas as pd

def find_repo_root(start=None, marker='smeft_contents.txt', max_up=8):
    p = Path(start or Path.cwd()).resolve()
    for _ in range(max_up+1):
        if (p / marker).exists():
            return p
        if p.parent == p:
            break
        p = p.parent
    raise FileNotFoundError(f'Could not find repo root containing {marker} from {Path.cwd()}')

REPO = find_repo_root()
NOTEBOOK_DIR = REPO / 'notebooks_hww_fit'
SMEFT_LISTING = REPO / 'smeft_contents.txt'
LOCAL_DICT = NOTEBOOK_DIR / 'dictionary.json'
print('repo:', REPO)
import importlib
P_ANALYSIS=Path('/uscms_data/d3/azhou/smeft/analysis')
HBB=P_ANALYSIS/'hbb-coffea'
for x in [P_ANALYSIS,HBB,NOTEBOOK_DIR]:
    if str(x) not in sys.path: sys.path.append(str(x))
import stxs_functions as sf
importlib.reload(sf)
manifest=json.loads((NOTEBOOK_DIR/'asset_manifest.json').read_text())

In [ ]:
# NOTE: sf.stxs_fit expects coffea/<name>.coffea in current working dir.
# If needed, create a symlink or copy selected file into REPO/coffea first.
print('default coffea path from manifest:', manifest.get('default_coffea'))

In [ ]:
def run_hbb_baseline(coffea_name, operators):
    wc_space = sf.stxs_fit(coffea_name, operators, verbose=False)
    rows=[]
    for k,v in wc_space.items():
        kk = k if isinstance(k, tuple) else (k,)
        rows.append({'wc':kk, 'chi2_total': float(v['total'][1]), 'xsec_total': float(v['total'][0])})
    df=pd.DataFrame(rows).sort_values('chi2_total')
    return df, wc_space

In [ ]:
# Example:
# df_hbb, wc_hbb = run_hbb_baseline('start0', ['cHW'])
# df_hbb.head()